# Cinema Backend — ma trận đo Pha 2 trên Kaggle

Notebook giữ công cụ đo trên nhánh hạ tầng và lần lượt build/chạy từng tag hoặc nhánh ứng dụng trong một checkout riêng. Mỗi phiên bản dùng schema PostgreSQL sạch; evidence ghi riêng commit ứng dụng và commit harness. Lượt mặc định là kiểm tra nhanh S1 ở 2 VU × 3 lần, chưa phải số liệu chính thức.

In [ ]:
import glob, json, os, pathlib, shutil, subprocess

REPO_URL = 'https://github.com/taclna/BTL-1-21-phase-2.git'
HARNESS_REF = 'feature/phase2-measurement-infrastructure'
APP_REFS = [
    'phase-1-baseline',
    'feature/phase2-idempotency',  # đổi thành phase-2-idempotency sau khi gắn tag
]
harness = pathlib.Path('/kaggle/working/phase2-harness')
app_checkout = pathlib.Path('/kaggle/working/phase2-app')

if not harness.exists():
    subprocess.run(['git', 'clone', '--branch', HARNESS_REF, '--single-branch',
                    REPO_URL, str(harness)], check=True)
else:
    subprocess.run(['git', 'fetch', 'origin', HARNESS_REF], cwd=harness, check=True)
    subprocess.run(['git', 'checkout', '-B', HARNESS_REF, 'origin/' + HARNESS_REF],
                   cwd=harness, check=True)

os.chdir(harness)
harness_commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
dirty = subprocess.check_output(
    ['git', 'status', '--porcelain', '--untracked-files=all', '--', 'src/main', 'pom.xml', 'load-test'],
    text=True,
).strip()
assert not dirty, 'Harness phải sạch trước khi đo:\n' + dirty
print('Harness:', HARNESS_REF, harness_commit)
print('Application refs:', APP_REFS)


In [ ]:
%%bash
set -euo pipefail
sudo apt-get update -qq
sudo apt-get install -y openjdk-21-jdk-headless postgresql postgresql-contrib nodejs curl ca-certificates gnupg
curl -fsSL https://dl.k6.io/key.gpg | sudo gpg --batch --yes --dearmor -o /usr/share/keyrings/k6-archive-keyring.gpg
echo 'deb [signed-by=/usr/share/keyrings/k6-archive-keyring.gpg] https://dl.k6.io/deb stable main' | sudo tee /etc/apt/sources.list.d/k6.list
sudo apt-get update -qq
sudo apt-get install -y k6


In [ ]:
jdk = sorted(glob.glob('/usr/lib/jvm/java-21-openjdk-*'))
assert jdk, 'Chưa cài được JDK 21'
os.environ['JAVA_HOME'] = jdk[0]
os.environ['PATH'] = jdk[0] + '/bin:' + os.environ['PATH']
subprocess.run(['java', '-version'], check=True)
subprocess.run(['k6', 'version'], check=True)
node_major = int(subprocess.check_output(
    ['node', '-p', 'process.versions.node.split(".")[0]'], text=True
))
assert node_major >= 18, 'Seed script cần Node.js >= 18'


In [ ]:
clusters = subprocess.check_output(['pg_lsclusters', '--no-header'], text=True).splitlines()
assert clusters, 'PostgreSQL chưa tạo cluster'
cluster = clusters[-1].split()
pg_version, pg_name, pg_port = cluster[:3]
subprocess.run(['sudo', 'pg_ctlcluster', '--skip-systemctl-redirect',
                pg_version, pg_name, 'start'], check=False)

def pg(sql):
    return subprocess.check_output(
        ['sudo', '-u', 'postgres', 'psql', '-p', pg_port,
         '-v', 'ON_ERROR_STOP=1', '-tAc', sql],
        text=True,
    ).strip()

pg("""DO $$ BEGIN
    IF NOT EXISTS (SELECT 1 FROM pg_roles WHERE rolname='cinema_load') THEN
        CREATE ROLE cinema_load LOGIN PASSWORD 'cinema_load_pw';
    END IF;
END $$;""")
if pg("SELECT 1 FROM pg_database WHERE datname='cinema_load'") != '1':
    subprocess.run(['sudo', '-u', 'postgres', 'createdb', '-p', pg_port,
                    '-O', 'cinema_load', 'cinema_load'], check=True)
data_dir = pg('SHOW data_directory')
database_pid = int(subprocess.check_output(
    ['sudo', 'head', '-n', '1', data_dir + '/postmaster.pid'], text=True
))
print('PostgreSQL:', pg('SELECT version()'))
print('Database PID:', database_pid, 'port:', pg_port)


In [ ]:
os.environ.update({
    'SPRING_PROFILES_ACTIVE': 'local',
    'DB_URL': f'jdbc:postgresql://127.0.0.1:{pg_port}/cinema_load',
    'DB_USERNAME': 'cinema_load',
    'DB_PASSWORD': 'cinema_load_pw',
    'PGHOST': '127.0.0.1',
    'PGPORT': str(pg_port),
    'PGDATABASE': 'cinema_load',
    'PGUSER': 'cinema_load',
    'PGPASSWORD': 'cinema_load_pw',
    'JWT_SECRET': 'kaggle-local-only-cinema-benchmark-secret-2026-please-change',
    'JWT_EXPIRATION_SECONDS': '14400',
    'DEMO_ADMIN_USERNAME': 'admin',
    'DEMO_ADMIN_PASSWORD': 'demo-pass-1234',
    'ADMIN_USERNAME': 'admin',
    'ADMIN_PASSWORD': 'demo-pass-1234',
    'DATABASE_PID': str(database_pid),
})
print('Environment configured for local load-test database')


In [ ]:
# Kiểm tra nhanh GĐ2: mỗi ref dùng DB sạch, 2 VU × 3 lượt contention.
command = [
    'python', 'load-test/run-matrix.py',
    '--repo-url', REPO_URL,
    '--app-dir', str(app_checkout),
    '--scenario', 'contention',
    '--levels', '2',
    '--repeats', '3',
    '--database-pid', str(database_pid),
    '--movie-count', '5',
    '--deleted-count', '0',
]
for app_ref in APP_REFS:
    command.extend(['--app-ref', app_ref])
subprocess.run(command, cwd=harness, env=os.environ.copy(), check=True)


In [ ]:
results_root = harness / 'load-test' / 'results'
matrix_dirs = sorted(results_root.glob('matrix-*'))
assert matrix_dirs, 'Chưa có kết quả matrix'
matrix_dir = matrix_dirs[-1]
manifest = json.loads((matrix_dir / 'matrix.json').read_text(encoding='utf-8'))
assert manifest['harnessGitCommit'] == harness_commit, manifest
assert [item['appRef'] for item in manifest['applications']] == APP_REFS, manifest
assert all(item['status'] == 'completed' for item in manifest['applications']), manifest

for item in manifest['applications']:
    environment_path = harness / item['resultDirectory'] / 'environment.json'
    evidence = json.loads(environment_path.read_text(encoding='utf-8'))
    assert evidence['appRef'] == item['appRef'], evidence
    assert evidence['appGitCommit'] == item['appGitCommit'], evidence
    assert evidence['harnessGitCommit'] == harness_commit, evidence
    print(item['appRef'], item['appGitCommit'], '→', item['resultDirectory'])

archive = shutil.make_archive(
    '/kaggle/working/cinema-phase2-matrix-results',
    'zip',
    root_dir=str(results_root),
)
print('Tải evidence tại Output:', archive)
